In [1]:
import sympy as sp

print("开始建立圆轨道竖杆动力学模型...")

# ============================================================
# 1. 定义符号变量
# ============================================================
# 广义坐标：q1 — 轨道角位置 (上视 CCW 正)；q2 — 法平面内倾角 (竖直=0, 向内为正)
q1, q2 = sp.symbols('q1 q2', real=True)
dq1, dq2 = sp.symbols('dq1 dq2', real=True)
ddq1, ddq2 = sp.symbols('ddq1 ddq2', real=True)

# 杆的物理参数
m = sp.symbols('m', real=True, positive=True)       # 质量
R = sp.symbols('R', real=True, positive=True)       # 圆轨道半径
l = sp.symbols('l', real=True, positive=True)       # 杆总长
a = sp.symbols('a', real=True, positive=True)       # 底部到质心距离

# 杆质心主轴转动惯量 (对角阵)
Ix, Iy, Iz = sp.symbols('Ix Iy Iz', real=True, positive=True)

# 等效重力向量 (g_app = g_world - a_base)
gx, gy, gz = sp.symbols('gx gy gz', real=True)
g_vec = sp.Matrix([gx, gy, gz])

# 状态向量 (2-DOF)
q = sp.Matrix([q1, q2])
dq = sp.Matrix([dq1, dq2])

# 简记
s1, c1 = sp.sin(q1), sp.cos(q1)
s2, c2 = sp.sin(q2), sp.cos(q2)

print("正在计算正运动学...")
# ============================================================
# 2. 正运动学 — 附着点与质心位置
# ============================================================
# 杆底部在圆轨道上的固定点 (xOy 平面)
p_attach = sp.Matrix([R*c1, R*s1, 0])

# 杆的方向向量 (竖直立起时 q2=0, 向内倾倒时 q2>0)
# z_body = 杆轴方向, y_body = 轨道切向 (也是 q2 的转轴)
e_rod = sp.Matrix([-s2*c1, -s2*s1, c2])

# 质心位置: p_cm = p_attach + a * e_rod
p_cm = sp.simplify(p_attach + a * e_rod)
print("p_cm ="); sp.pprint(p_cm)

print("\n正在计算平动与转动速度...")
# ============================================================
# 3. 速度与动能
# ============================================================
# 平动速度 (雅可比法)
v_cm = p_cm.jacobian(q) * dq
v_cm = sp.simplify(v_cm)

# 体坐标系 (body frame): Xb 正交法平面分量, Yb 切向(=q2转轴), Zb=杆轴
y_body = sp.Matrix([s1, -c1, 0])          # 轨道切向
z_body = e_rod                             # 杆轴方向
x_body = y_body.cross(z_body)             # 法平面内正交方向
R_body = sp.Matrix.hstack(x_body, y_body, z_body)

# 全局角速度: 绕 Z 轴的 q1 + 绕切向轴的 q2
omega_global = sp.Matrix([dq2*s1, -dq2*c1, dq1])

# 转换到体坐标系 (以使用对角化惯量矩阵)
omega_body = R_body.T * omega_global
omega_body = sp.simplify(omega_body)
print("omega_body ="); sp.pprint(omega_body)

# 惯量矩阵
I_body = sp.diag(Ix, Iy, Iz)

# 动能 (平动 + 转动)
T_trans = 0.5 * m * v_cm.dot(v_cm)
T_rot = 0.5 * (omega_body.T * I_body * omega_body)[0]
T = sp.simplify(T_trans + T_rot)

# ============================================================
# 4. 势能 (V = -m * g_vec^T * p_cm)
# ============================================================
print("正在计算势能...")
V = -m * g_vec.dot(p_cm)
V = sp.simplify(V)

# ============================================================
# 5. 提取质量矩阵 M(q) — 2x2
# ============================================================
print("正在推导质量矩阵 M...")
M = sp.zeros(2, 2)
for i in range(2):
    for j in range(2):
        M[i, j] = sp.diff(sp.diff(T, dq[i]), dq[j])
M = sp.simplify(M)

print("M ="); sp.pprint(M)

# ============================================================
# 6. 提取科里奥利与离心矩阵 C(q, dq) — Christoffel 符号
# ============================================================
print("\n正在推导科里奥利矩阵 C...")
C = sp.zeros(2, 2)
for k in range(2):
    for j in range(2):
        c_val = 0
        for i in range(2):
            c_ijk = 0.5 * (sp.diff(M[k, j], q[i]) +
                           sp.diff(M[k, i], q[j]) -
                           sp.diff(M[i, j], q[k]))
            c_val += c_ijk * dq[i]
        C[k, j] = c_val
C = sp.simplify(C)

print("C ="); sp.pprint(C)

# ============================================================
# 7. 提取重力项 G(q) — 2x1
# ============================================================
print("\n正在推导重力项 G...")
G = sp.zeros(2, 1)
for i in range(2):
    G[i] = sp.diff(V, q[i])
G = sp.simplify(G)

print("G ="); sp.pprint(G)

# ============================================================
# 8. 验证 — 对称性与反对称性
# ============================================================
print("\n======== 验证 ========")
print(f"M 对称: {sp.simplify(M - M.T).is_zero_matrix}")

# Ṁ - 2C 反对称性
M_dot = sp.zeros(2, 2)
for i in range(2):
    for j in range(2):
        M_dot[i, j] = sp.diff(M[i, j], q1)*dq1 + sp.diff(M[i, j], q2)*dq2
skew_check = sp.simplify(M_dot - C - C.T)
print(f"dM/dt - C - C^T 反对称: {skew_check.is_zero_matrix}")

# ============================================================
# 9. 导出 C 代码
# ============================================================
print("\n======== C 代码生成 ========")
print("\n// --- 质量矩阵 M ---")
for i in range(2):
    for j in range(2):
        print(sp.ccode(M[i, j], assign_to=f"M[{i}][{j}]"))

print("\n// --- 科里奥利矩阵 C ---")
for i in range(2):
    for j in range(2):
        print(sp.ccode(C[i, j], assign_to=f"C[{i}][{j}]"))

print("\n// --- 重力向量 G ---")
for i in range(2):
    print(sp.ccode(G[i, 0], assign_to=f"G[{i}]"))


开始建立圆轨道竖杆动力学模型...
正在计算正运动学...
p_cm =
⎡(R - a⋅sin(q₂))⋅cos(q₁)⎤
⎢                       ⎥
⎢(R - a⋅sin(q₂))⋅sin(q₁)⎥
⎢                       ⎥
⎣       a⋅cos(q₂)       ⎦

正在计算平动与转动速度...
omega_body =
⎡-dq₁⋅sin(q₂)⎤
⎢            ⎥
⎢    dq₂     ⎥
⎢            ⎥
⎣dq₁⋅cos(q₂) ⎦
正在计算势能...
正在推导质量矩阵 M...
M =
⎡          2                 2                             2                   ⎤
⎢1.0⋅Ix⋅sin (q₂) + 1.0⋅Iz⋅cos (q₂) + 1.0⋅m⋅(-R + a⋅sin(q₂))           0        ⎥
⎢                                                                              ⎥
⎢                                                                           2  ⎥
⎣                             0                               1.0⋅Iy + 1.0⋅a ⋅m⎦

正在推导科里奥利矩阵 C...
C =
⎡-1.0⋅dq₂⋅(-Ix⋅sin(q₂) + Iz⋅sin(q₂) + a⋅m⋅(R - a⋅sin(q₂)))⋅cos(q₂)  -1.0⋅dq₁⋅( ↪
⎢                                                                              ↪
⎣1.0⋅dq₁⋅(-Ix⋅sin(q₂) + Iz⋅sin(q₂) + a⋅m⋅(R - a⋅sin(q₂)))⋅cos(q₂)              ↪

↪ -Ix⋅sin(q₂) + Iz⋅sin(q₂) + a⋅

In [2]:
import numpy as np

print("=== 数值 SPOT CHECK: 与手算公式对照 ===")

# --- 数值参数 ---
m_val = 1.0          # kg
R_val = 1.0          # m (轨道半径)
l_val = 2.0          # m (杆长)
a_val = 0.5          # m (底部到质心)

# 细杆惯量: Iz≈0, Ix=Iy=m*l²/12
Ix_val = m_val * l_val**2 / 12
Iy_val = Ix_val
Iz_val = 0.0

# 重力
g0 = 9.8
gx_val, gy_val, gz_val = 0.0, 0.0, -g0

# 测试构型
q1_val = np.deg2rad(30)    # 30°
q2_val = np.deg2rad(15)    # 15° 向内倾倒
dq1_val = 1.0
dq2_val = 0.5

s1, c1 = np.sin(q1_val), np.cos(q1_val)
s2, c2 = np.sin(q2_val), np.cos(q2_val)

# ---- 手算 M ----
M11_hand = m_val * (R_val - a_val*s2)**2 + Ix_val*s2**2 + Iz_val*c2**2
M22_hand = m_val * a_val**2 + Iy_val
print(f"\nM11: sym={M11_hand:.6f}")
print(f"M22: sym={M22_hand:.6f}")
print(f"M12=M21=0")

# ---- 手算 C ----
alpha = -m_val*a_val*c2*(R_val - a_val*s2) + s2*c2*(Ix_val - Iz_val)
C11_hand = alpha * dq2_val
C12_hand = alpha * dq1_val
C21_hand = -alpha * dq1_val
C22_hand = 0.0
print(f"\nalpha = {alpha:.6f}")
print(f"C11 = alpha*dq2 = {C11_hand:.6f}")
print(f"C12 = alpha*dq1 = {C12_hand:.6f}")
print(f"C21 = -alpha*dq1 = {C21_hand:.6f}")
print(f"C22 = 0")

# ---- 手算 G ----
G1_hand = m_val * (R_val - a_val*s2) * (gx_val*s1 - gy_val*c1)
G2_hand = m_val*a_val*c2*(gx_val*c1 + gy_val*s1) + m_val*gz_val*a_val*s2
print(f"\nG1 = {G1_hand:.6f} (q1 方向力矩应为 0，因 gx=gy=0)")
print(f"G2 = {G2_hand:.6f}")

# ---- λ 函数评估 Sympy 结果做交叉验证 ----
M_func = sp.lambdify((q1,q2, m,R,a, Ix,Iy,Iz), M, "numpy")
C_func = sp.lambdify((q1,q2,dq1,dq2, m,R,a, Ix,Iy,Iz), C, "numpy")
G_func = sp.lambdify((q1,q2, m,R,a, gx,gy,gz), G, "numpy")

M_num = np.array(M_func(q1_val,q2_val, m_val,R_val,a_val, Ix_val,Iy_val,Iz_val))
C_num = np.array(C_func(q1_val,q2_val,dq1_val,dq2_val, m_val,R_val,a_val, Ix_val,Iy_val,Iz_val))
G_num = np.array(G_func(q1_val,q2_val, m_val,R_val,a_val, gx_val,gy_val,gz_val)).flatten()

print("\n--- Sympy lambdify 评估 ---")
print("M =\n", M_num)
print("C =\n", C_num)
print("G =", G_num)

# 对照
assert np.allclose(M_num[0,0], M11_hand), "M11 mismatch!"
assert np.allclose(M_num[1,1], M22_hand), "M22 mismatch!"
assert np.allclose(C_num[0,0], C11_hand), "C11 mismatch!"
assert np.allclose(C_num[0,1], C12_hand), "C12 mismatch!"
assert np.allclose(C_num[1,0], C21_hand), "C21 mismatch!"
assert np.allclose(G_num[0], G1_hand), "G1 mismatch!"
assert np.allclose(G_num[1], G2_hand), "G2 mismatch!"
print("\n✓ 全部通过: Sympy 符号解与手算公式完全一致")


=== 数值 SPOT CHECK: 与手算公式对照 ===

M11: sym=0.780257
M22: sym=0.583333
M12=M21=0

alpha = -0.337130
C11 = alpha*dq2 = -0.168565
C12 = alpha*dq1 = -0.337130
C21 = -alpha*dq1 = 0.337130
C22 = 0

G1 = 0.000000 (q1 方向力矩应为 0，因 gx=gy=0)
G2 = -1.268213

--- Sympy lambdify 评估 ---
M =
 [[0.78025688 0.        ]
 [0.         0.58333333]]
C =
 [[-0.16856479 -0.33712958]
 [ 0.33712958  0.        ]]
G = [ 0.         -1.26821332]

✓ 全部通过: Sympy 符号解与手算公式完全一致


In [6]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

print("=== 重力回归矩阵 Y(q, g_vec) ===")
# G₁ = m·(R − a·s₂)·(gx·s₁ − gy·c₁)
#    = mR·(gx·s₁−gy·c₁) − ma·s₂·(gx·s₁−gy·c₁)
# G₂ = ma·(gx·c₁c₂ + gy·s₁c₂ + gz·s₂)
# → 线性组合参数: θ₁ = m·R, θ₂ = m·a

theta1, theta2 = sp.symbols('theta1 theta2', real=True)

Y_sym = sp.zeros(2, 2)
Y_sym[:, 0] = sp.simplify(sp.diff(G, R) / m)   # ∂G/∂R / m → θ₁ 的系数
Y_sym[:, 1] = sp.simplify(sp.diff(G, a) / m)   # ∂G/∂a / m → θ₂ 的系数

print("Y(2×2) [θ₁=mR, θ₂=ma]:")
sp.pprint(Y_sym)

# 验证
assert sp.simplify((Y_sym * sp.Matrix([m*R, m*a])) - G).is_zero_matrix, "Y·θ ≠ G!"
print("✓ Y·[mR, ma]ᵀ == G")

# ---- 数值辨识函数 ----
Y_func = sp.lambdify((q1, q2, gx, gy, gz), Y_sym, "numpy")

def gravity_regressor_2d(q_val, g_val):
    """Y(2×2): q_val=[q1,q2], g_val=[gx,gy,gz]"""
    return np.array(Y_func(*q_val, *g_val), dtype=float).reshape(2, 2)

def identify_gravity_params(q_data, g_data, tau_g_data):
    """
    静态重力参数辨识
    q_data:  (N,2)  [q1, q2]
    g_data:  (N,3)  [gx, gy, gz]
    tau_g_data: (N,2)  重力矩测量值
    返回: theta_hat = [mR, ma], R²
    """
    N = len(q_data)
    Y_stack = np.vstack([gravity_regressor_2d(q_data[i], g_data[i]) for i in range(N)])
    reg = LinearRegression(fit_intercept=False).fit(Y_stack, tau_g_data.reshape(-1))
    return reg.coef_, reg.score(Y_stack, tau_g_data.reshape(-1))


=== 重力回归矩阵 Y(q, g_vec) ===
Y(2×2) [θ₁=mR, θ₂=ma]:
⎡gx⋅sin(q₁) - gy⋅cos(q₁)           (-gx⋅sin(q₁) + gy⋅cos(q₁))⋅sin(q₂)         ⎤
⎢                                                                             ⎥
⎣           0             gx⋅cos(q₁)⋅cos(q₂) + gy⋅sin(q₁)⋅cos(q₂) + gz⋅sin(q₂)⎦
✓ Y·[mR, ma]ᵀ == G


In [7]:
print("=== 完整动力学模型参数辨识 (M + C + G) ===")
# τ = M(q)·ddq + C(q,dq)·dq + G(q)   →  线性参数化

# ---- 组合参数定义 ----
# β₁ = mR² + Iz        M₁₁ 常数项
# β₂ = −2mRa           M₁₁·s₂ 项, C 的 c₂ 项
# β₃ = ma² + Ix − Iz   M₁₁·s₂² 项, C 的 s₂c₂ 项
# β₄ = Iy + ma²         M₂₂
# θ₂ = ma              G₂ (及 G₁ 中 a 相关项)
# θ₁ = mR              G₁ (平放数据 G₁≡0, 需倾斜平台; 可跳过)

# ---- 构造逐行回归矩阵 Y_dyn(2×5) ----
# 参数向量: [β₁, β₂, β₃, β₄, θ₂]
#
# τ₁ = M₁₁·ddq₁ + C₁₁·dq₁ + C₁₂·dq₂ + G₁
#    = (β₁+β₂·s₂+β₃·s₂²)·ddq₁ + 2α·dq₁·dq₂ + G₁
#    其中 α = (β₂/2)·c₂ + β₃·s₂c₂
# → τ₁ = β₁·ddq₁ + β₂·(s₂·ddq₁ + c₂·dq₁·dq₂)
#        + β₃·(s₂²·ddq₁ + 2s₂c₂·dq₁·dq₂)
#        + θ₁·(gx·s₁−gy·c₁) − θ₂·s₂·(gx·s₁−gy·c₁)
#
# τ₂ = M₂₂·ddq₂ + C₂₁·dq₁ + C₂₂·dq₂ + G₂
#    = β₄·ddq₂ − α·dq₁² + G₂
# → τ₂ = β₂·(−c₂·dq₁²/2) + β₃·(−s₂c₂·dq₁²) + β₄·ddq₂
#        + θ₂·(gx·c₁c₂+gy·s₁c₂+gz·s₂)

s2, c2 = sp.sin(q2), sp.cos(q2)
s1, c1 = sp.sin(q1), sp.cos(q1)

# ---- 用已推导的 M, C, G 符号解导出回归列 ----
# τ₁ = M₁₁·ddq₁ + C₁₁·dq₁ + C₁₂·dq₂ + G₁
# τ₂ = M₂₂·ddq₂ + C₂₁·dq₁ + C₂₂·dq₂ + G₂

# 用已推导的 M, C, G 符号解提取回归列
beta1, beta2, beta3, beta4 = sp.symbols('beta1 beta2 beta3 beta4', real=True)
theta2_id = sp.symbols('theta2_id', real=True)

# 将 M, C, G 用 β 参数重写
# M₁₁ = β₁ + β₂·s₂ + β₃·s₂²
# M₂₂ = β₄
# α = (β₂/2)·c₂ + β₃·s₂c₂
# G₁ = θ₁·(gx·s₁−gy·c₁) − θ₂·s₂·(gx·s₁−gy·c₁)
# G₂ = θ₂·(gx·c₁c₂+gy·s₁c₂+gz·s₂)

alpha_expr = (beta2/2)*c2 + beta3*s2*c2
M11_expr = beta1 + beta2*s2 + beta3*s2**2
M22_expr = beta4

# τ₁ 表达式
tau1_expr = (M11_expr*ddq1 + 2*alpha_expr*dq1*dq2
             + theta1*(gx*s1 - gy*c1) - theta2_id*s2*(gx*s1 - gy*c1))
tau2_expr = (M22_expr*ddq2 - alpha_expr*dq1**2
             + theta2_id*(gx*c1*c2 + gy*s1*c2 + gz*s2))

# 对每个参数求偏导得到回归列 (2×1 列向量)
reg_beta1  = sp.simplify(sp.Matrix([sp.diff(tau1_expr, beta1), sp.diff(tau2_expr, beta1)]))
reg_beta2  = sp.simplify(sp.Matrix([sp.diff(tau1_expr, beta2), sp.diff(tau2_expr, beta2)]))
reg_beta3  = sp.simplify(sp.Matrix([sp.diff(tau1_expr, beta3), sp.diff(tau2_expr, beta3)]))
reg_beta4  = sp.simplify(sp.Matrix([sp.diff(tau1_expr, beta4), sp.diff(tau2_expr, beta4)]))
reg_theta2 = sp.simplify(sp.Matrix([sp.diff(tau1_expr, theta2_id), sp.diff(tau2_expr, theta2_id)]))

print("回归列 (2×1 per parameter):")
for name, col in [("β₁", reg_beta1), ("β₂", reg_beta2), ("β₃", reg_beta3), ("β₄", reg_beta4), ("θ₂", reg_theta2)]:
    print(f"\n--- ∂τ/∂{name} ---")
    sp.pprint(col)

# ---- 数值辨识函数 ----
# lambdify 所有回归列
reg_funcs = [
    sp.lambdify((q1,q2, dq1,dq2, ddq1,ddq2, gx,gy,gz), col, "numpy")
    for col in [reg_beta1, reg_beta2, reg_beta3, reg_beta4, reg_theta2]
]

def dynamics_regressor(state, g_vec):
    """
    state: [q1, q2, dq1, dq2, ddq1, ddq2]
    g_vec: [gx, gy, gz]
    返回 Y_dyn(2×5)
    """
    args = (*state, *g_vec)
    return np.column_stack([np.array(f(*args), dtype=float).flatten() for f in reg_funcs])

def identify_full_dynamics(state_data, g_data, tau_data):
    """
    完整动力学参数辨识 (需动态激励轨迹)
    state_data: (N,6)  [q1,q2,dq1,dq2,ddq1,ddq2]
    g_data:     (N,3)  [gx,gy,gz]
    tau_data:   (N,2)  力矩测量值
    返回: params = [β₁,β₂,β₃,β₄,θ₂], R²
    """
    N = len(state_data)
    Y_stack = np.vstack([dynamics_regressor(state_data[i], g_data[i]) for i in range(N)])
    reg = LinearRegression(fit_intercept=False).fit(Y_stack, tau_data.reshape(-1))
    return reg.coef_, reg.score(Y_stack, tau_data.reshape(-1))


print("\n\n=== 合成数据验证 (动态激励轨迹) ===")
# 用标称参数生成含噪声动态数据验证辨识

# --- 标称物理参数 ---
m_true = 0.5
R_true = 0.6
a_true = 0.2
l_true = 0.8
Ix_true = m_true * l_true**2 / 12
Iy_true = Ix_true
Iz_true = 0.0

# 对应的组合参数真值
beta_true = np.array([
    m_true*R_true**2 + Iz_true,     # β₁
    -2*m_true*R_true*a_true,        # β₂
    m_true*a_true**2 + Ix_true - Iz_true,  # β₃
    Iy_true + m_true*a_true**2,     # β₄
    m_true * a_true                 # θ₂
])

g0 = 9.8
gx_nom, gy_nom, gz_nom = 0.0, 0.0, -g0  # 平放

# --- 生成激励轨迹 (N 个时间点) ---
np.random.seed(123)
N = 500
t = np.linspace(0, 10, N)
dt = t[1] - t[0]

# q1: 扫频正弦, q2: 叠加正弦
q1_traj = np.sin(0.8*t) + 0.3*np.sin(2.3*t)
q2_traj = 0.4*np.sin(1.5*t) + 0.15*np.sin(3.7*t)
dq1_traj = np.gradient(q1_traj, dt)
dq2_traj = np.gradient(q2_traj, dt)
ddq1_traj = np.gradient(dq1_traj, dt)
ddq2_traj = np.gradient(dq2_traj, dt)

state_data = np.column_stack([q1_traj, q2_traj, dq1_traj, dq2_traj, ddq1_traj, ddq2_traj])
g_data = np.tile([gx_nom, gy_nom, gz_nom], (N, 1))

# 用标称参数计算理论力矩
M_func = sp.lambdify((q1,q2, m,R,a, Ix,Iy,Iz), M, "numpy")
C_func = sp.lambdify((q1,q2,dq1,dq2, m,R,a, Ix,Iy,Iz), C, "numpy")
G_func = sp.lambdify((q1,q2, m,R,a, gx,gy,gz), G, "numpy")

tau_true = np.zeros((N, 2))
for i in range(N):
    Mi = np.array(M_func(q1_traj[i], q2_traj[i], m_true, R_true, a_true, Ix_true, Iy_true, Iz_true))
    Ci = np.array(C_func(q1_traj[i], q2_traj[i], dq1_traj[i], dq2_traj[i], m_true, R_true, a_true, Ix_true, Iy_true, Iz_true))
    Gi = np.array(G_func(q1_traj[i], q2_traj[i], m_true, R_true, a_true, gx_nom, gy_nom, gz_nom)).flatten()
    ddq = np.array([ddq1_traj[i], ddq2_traj[i]])
    dq = np.array([dq1_traj[i], dq2_traj[i]])
    tau_true[i] = Mi @ ddq + Ci @ dq + Gi

# 加噪声
tau_meas = tau_true + np.random.randn(N, 2) * 0.01

# --- 辨识 ---
params_hat, r2 = identify_full_dynamics(state_data, g_data, tau_meas)

print(f"标称真值: β₁={beta_true[0]:.4f}, β₂={beta_true[1]:.4f}, β₃={beta_true[2]:.4f}, β₄={beta_true[3]:.4f}, θ₂={beta_true[4]:.4f}")
print(f"辨识结果: β̂₁={params_hat[0]:.4f}, β̂₂={params_hat[1]:.4f}, β̂₃={params_hat[2]:.4f}, β̂₄={params_hat[3]:.4f}, θ̂₂={params_hat[4]:.4f}")
print(f"R² = {r2:.6f}")

# --- 反解物理参数 ---
# 已知 m, R (由 CAD/称重): 
# θ₂=ma  → â = θ̂₂/m
# β₂=−2mRa → 验证一致性: −β̂₂/(2m) / â ≈ R ?
# β₁=mR²+Iz → Îz = β̂₁ − mR²
# β₃=ma²+Ix−Iz → Îx = β̂₃ − mâ² + Îz
# β₄=Iy+ma² → Îy = β̂₄ − mâ²

a_hat = params_hat[4] / m_true
Iz_hat = params_hat[0] - m_true * R_true**2
Ix_hat = params_hat[2] - m_true * a_hat**2 + Iz_hat
Iy_hat = params_hat[3] - m_true * a_hat**2

print(f"\n--- 反解物理参数 (已知 m={m_true}, R={R_true}) ---")
print(f"â = {a_hat:.4f}  (真值: {a_true})")
print(f"Îx = {Ix_hat:.6f}  (真值: {Ix_true:.6f})")
print(f"Îy = {Iy_hat:.6f}  (真值: {Iy_true:.6f})")
print(f"Îz = {Iz_hat:.6f}  (真值: {Iz_true})")


=== 完整动力学模型参数辨识 (M + C + G) ===
回归列 (2×1 per parameter):

--- ∂τ/∂β₁ ---
⎡ddq₁⎤
⎢    ⎥
⎣ 0  ⎦

--- ∂τ/∂β₂ ---
⎡ddq₁⋅sin(q₂) + dq₁⋅dq₂⋅cos(q₂)⎤
⎢                              ⎥
⎢            2                 ⎥
⎢        -dq₁ ⋅cos(q₂)         ⎥
⎢        ──────────────        ⎥
⎣              2               ⎦

--- ∂τ/∂β₃ ---
⎡  ddq₁⋅cos(2⋅q₂)   ddq₁                    ⎤
⎢- ────────────── + ──── + dq₁⋅dq₂⋅sin(2⋅q₂)⎥
⎢        2           2                      ⎥
⎢                                           ⎥
⎢                 2                         ⎥
⎢             -dq₁ ⋅sin(2⋅q₂)               ⎥
⎢             ────────────────              ⎥
⎣                    2                      ⎦

--- ∂τ/∂β₄ ---
⎡ 0  ⎤
⎢    ⎥
⎣ddq₂⎦

--- ∂τ/∂θ₂ ---
⎡         (-gx⋅sin(q₁) + gy⋅cos(q₁))⋅sin(q₂)         ⎤
⎢                                                    ⎥
⎣gx⋅cos(q₁)⋅cos(q₂) + gy⋅sin(q₁)⋅cos(q₂) + gz⋅sin(q₂)⎦


=== 合成数据验证 (动态激励轨迹) ===
标称真值: β₁=0.1800, β₂=-0.1200, β₃=0.0467, β₄=0.0467, θ₂=0.1000
辨识结